# Lecture 6 — Class Exercise
## Part-to-Whole: Hierarchical Visualization

> **Push to:** `week06/lecture06_exercise.ipynb`

**Rules:**
1. Use `px` first, then customise with `update_traces` / `update_layout`
2. Colour encodes a meaningful category — not decoration
3. Insight title names the specific finding
4. Consider: would a bar chart be clearer? If yes, use the bar chart

---


In [1]:
import pandas as pd
import plotly.express as px
import numpy as np

# Dataset: Global Energy Mix by Country and Source
df = pd.read_csv('/content/global_energy_mix.csv')

# Source type mapping — reuse from lecture
source_category = {
    'Coal': 'Fossil', 'Oil': 'Fossil', 'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon', 'Hydro': 'Low-carbon',
    'Wind': 'Renewable', 'Solar': 'Renewable', 'Other Renewables': 'Renewable'
}
df['Source_Type'] = df['Source'].map(source_category)

print(f"Loaded: {len(df)} rows")
print(df.head(10))


Loaded: 103 rows
         Country         Region            Source  Share_pct     TWh  \
0  United States  North America              Coal         10  1015.0   
1  United States  North America               Oil         35  3220.0   
2  United States  North America       Natural Gas         34  3083.0   
3  United States  North America           Nuclear          9   798.0   
4  United States  North America             Hydro          3   339.0   
5  United States  North America              Wind          4   413.0   
6  United States  North America             Solar          3   325.0   
7  United States  North America  Other Renewables          2   229.0   
8          China           Asia              Coal         58  6929.1   
9          China           Asia               Oil         18  1620.0   

  Source_Type  
0      Fossil  
1      Fossil  
2      Fossil  
3  Low-carbon  
4  Low-carbon  
5   Renewable  
6   Renewable  
7   Renewable  
8      Fossil  
9      Fossil  


## Task 1 — Treemap: fossil fuel dependency by country

**What to build:** A treemap showing **fossil fuel TWh only**, broken down by Region → Country → Source (Coal / Oil / Natural Gas).

**Requirements:**
- Filter to fossil sources only before plotting
- Use `path=['Region', 'Country', 'Source']` for the hierarchy
- Colour encodes the fossil source type (Coal / Oil / Natural Gas) with a CVD-safe palette
- Show TWh values in labels — no percentages
- Grey out parent nodes (Region and Country level)
- Insight title naming which region or country is most fossil-dependent

> 💡 `df.loc[df['Source_Type'] == 'Fossil']`


In [2]:
# Task 1
# -------
fossil = df.loc[df['Source_Type'] == 'Fossil'].copy()

fossil_colors = {'Coal': '#0072B2', 'Oil': '#D55E00', 'Natural Gas': '#009E73'}

fig1 = px.treemap(
    fossil,
    path=['Region', 'Country', 'Source'],
    values='TWh',
    color='Source',
    color_discrete_map=fossil_colors,
)

# Grey out parent nodes (Region, Country level) — only leaf (Source) nodes keep their colour
leaf_labels = set(fossil['Source'].unique())
colors = list(fig1.data[0].marker.colors)
labels = list(fig1.data[0].labels)
new_colors = [c if lbl in leaf_labels else '#BBBBBB' for lbl, c in zip(labels, colors)]
fig1.data[0].marker.colors = new_colors

fig1.update_traces(
    texttemplate='%{label}<br>%{value:.0f} TWh',
    textfont=dict(family='Arial, sans-serif', size=12),
)

fossil_by_country = fossil.groupby('Country')['TWh'].sum()
fossil_share = fossil.groupby('Country')['Share_pct'].sum().sort_values(ascending=False)
top_country = fossil_share.index[0]
top_share = fossil_share.iloc[0]

fig1.update_layout(
    title=dict(
        text=(f"<b>{top_country} is almost entirely fossil-dependent, at {top_share:.0f}% of its energy mix</b>"
              f"<br><sup>Fossil fuel generation (TWh) by region, country, and source</sup>"),
        x=0, xanchor='left', font=dict(size=16)
    ),
    font=dict(family='Arial, sans-serif', size=13, color='#333'),
    margin=dict(l=10, r=10, t=90, b=10),
    height=550,
    width=900,
)

fig1.show()


## Task 2 — Sunburst: tipping behaviour by day and meal time

**What to build:** A sunburst chart using the built-in `tips` dataset showing how **total bill amount** is distributed across day → time → smoker status.

**Requirements:**
- Load tips with `px.data.tips()`
- Aggregate **total bill** (sum of `total_bill`) per group — not count
- Hierarchy: `path=['day', 'time', 'smoker']`
- Colour encodes smoker status with a CVD-safe blue/orange palette
- Grey out parent nodes (day and time level)
- Use `percent parent` for text labels
- Insight title describing where the most spending happens

> 💡 `tips.groupby(['day', 'time', 'smoker'])['total_bill'].sum().reset_index()`


In [3]:
# Task 2
# -------
tips = px.data.tips()

tips_agg = tips.groupby(['day', 'time', 'smoker'])['total_bill'].sum().reset_index()

smoker_colors = {'Yes': '#E69F00', 'No': '#0072B2'}

fig2 = px.sunburst(
    tips_agg,
    path=['day', 'time', 'smoker'],
    values='total_bill',
    color='smoker',
    color_discrete_map=smoker_colors,
)

# Grey out parent nodes (day, time level) — only leaf (smoker) nodes keep their colour
leaf_labels = set(tips_agg['smoker'].unique())
colors = list(fig2.data[0].marker.colors)
labels = list(fig2.data[0].labels)
new_colors = [c if lbl in leaf_labels else '#BBBBBB' for lbl, c in zip(labels, colors)]
fig2.data[0].marker.colors = new_colors

fig2.update_traces(
    textinfo='percent parent',
    textfont=dict(family='Arial, sans-serif', size=12),
)

top_group = tips_agg.sort_values('total_bill', ascending=False).iloc[0]

fig2.update_layout(
    title=dict(
        text=(f"<b>{top_group['day']} {top_group['time']} ({top_group['smoker']} smokers) drives the most total spending</b>"
              f"<br><sup>Total bill amount ($) by day, time, and smoker status</sup>"),
        x=0, xanchor='left', font=dict(size=16)
    ),
    font=dict(family='Arial, sans-serif', size=13, color='#333'),
    margin=dict(l=10, r=10, t=90, b=10),
    height=550,
    width=900,
)

fig2.show()


## Task 3 — Treemap vs bar: low-carbon energy by country

**What to build:** Build **both** a treemap and a horizontal bar chart showing total low-carbon TWh (Nuclear + Hydro) per country. Then answer the question in a markdown cell below.

**Requirements:**
- Filter to `Source_Type == 'Low-carbon'` and aggregate TWh by country
- Treemap: single-level `path=['All', 'Country']` with a dummy root node labelled `'Low-carbon'`
- Bar chart: sorted by TWh, horizontal orientation, CVD-safe colour
- Both charts show TWh values, not percentages
- Insight title on the bar chart naming the leading country


In [4]:
# Task 3 — charts
# -----------------
import plotly.graph_objects as go

low_carbon = df.loc[df['Source_Type'] == 'Low-carbon'].copy()
low_carbon_by_country = (low_carbon.groupby('Country')['TWh']
                          .sum()
                          .reset_index()
                          .sort_values('TWh', ascending=False))

leading_country = low_carbon_by_country.iloc[0]['Country']
leading_twh = low_carbon_by_country.iloc[0]['TWh']

# --- Treemap version ---
low_carbon_by_country['All'] = 'Low-carbon'

fig3a = px.treemap(
    low_carbon_by_country,
    path=['All', 'Country'],
    values='TWh',
    color_discrete_sequence=['#BBBBBB'],
)
fig3a.update_traces(
    texttemplate='%{label}<br>%{value:.0f} TWh',
    textfont=dict(family='Arial, sans-serif', size=12),
)
# Root node stays grey; colour the country leaves for readability
fig3a.data[0].marker.colors = ['#BBBBBB'] + ['#0072B2'] * (len(low_carbon_by_country))

fig3a.update_layout(
    title=dict(text='<b>Low-carbon generation (Nuclear + Hydro) by country</b>', x=0, xanchor='left', font=dict(size=15)),
    font=dict(family='Arial, sans-serif', size=13, color='#333'),
    margin=dict(l=10, r=10, t=70, b=10),
    height=450,
    width=850,
)
fig3a.show()

# --- Bar chart version ---
bar_order = low_carbon_by_country.sort_values('TWh')

fig3b = go.Figure(go.Bar(
    x=bar_order['TWh'],
    y=bar_order['Country'],
    orientation='h',
    marker_color='#0072B2',
    text=bar_order['TWh'].round(0),
    textposition='outside',
    cliponaxis=False,
))

fig3b.update_layout(
    title=dict(
        text=(f"<b>{leading_country} leads the world in low-carbon energy, generating {leading_twh:.0f} TWh</b>"
              f"<br><sup>Nuclear + Hydro generation (TWh) by country</sup>"),
        x=0, xanchor='left', font=dict(size=16)
    ),
    xaxis=dict(title='TWh', range=[0, bar_order['TWh'].max() * 1.15], showgrid=True, gridcolor='#eeeeee', zeroline=False),
    yaxis=dict(title=''),
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial, sans-serif', size=13, color='#333'),
    height=500,
    width=850,
    margin=dict(l=10, r=60, t=90, b=40),
)
fig3b.show()


**Which is clearer — the treemap or the bar chart?**

The **bar chart** is clearer here. This is a single-level comparison (one value per country, no real hierarchy — the `'Low-carbon'` root in the treemap is just a dummy label), and bar charts are far better than treemaps at letting readers compare and rank values precisely: aligned baselines and lengths are easy to compare, while treemap rectangle *areas* are hard to compare accurately by eye, especially for close values.

The treemap would earn its place if there were a genuine multi-level hierarchy to show (e.g. Region → Country → Source, as in Task 1), where part-to-whole and nesting matter more than precise ranking. For a flat one-level "which country generates the most" question, the bar chart wins on both clarity and precision.
